# Data Science Beginner's Guide: Detecting - Skewness, High Correlation, Target Bias (Class Imbalance), and Long-Tailed Categorical Values

This guide walks you through creating a 100-row simulated dataset in Python that intentionally contains four common real-world data issues: **Skewness**, **High Correlation**, **Target Bias (Class Imbalance)**, and **Long-Tailed Categorical Values**. You will learn how to identify each issue using metrics and how to apply standard beginner-friendly fixes.

---

## 🛠️ Step 1: Create the Simulated Dataset

Copy and run this block first to generate your 100-row DataFrame containing the intentional flaws.

```python
import pandas as pd
import numpy as np

# Set a random seed for reproducible results
np.random.seed(42)
n = 100

# 1. Skewed Variable: Income (Log-normal distribution creates extreme high values)
income = np.random.lognormal(mean=10, sigma=1.0, size=n)

# 2. Correlated Variables: Performance score is strongly linear with Experience Years
experience = np.random.randint(1, 25, size=n)
performance = (experience * 3.5) + np.random.normal(0, 5, size=n)

# 3. Biased Target: Over-representation (90% Approved / 10% Rejected)
loan_status = np.random.choice(['Approved', 'Rejected'], size=n, p=[0.90, 0.10])

# 4. Long-Tailed Category: Top 3 cities dominate; 20 small towns split the tiny remainder
cities = ['New York', 'Los Angeles', 'Chicago'] + [f'Town_{i}' for i in range(20)]
city_probabilities = [0.5, 0.2, 0.1] + [0.2/20] * 20
chosen_cities = np.random.choice(cities, size=n, p=city_probabilities)

# Combine everything into a single DataFrame
df = pd.DataFrame({
    'Income': income,
    'Experience_Years': experience,
    'Performance_Score': performance,
    'City': chosen_cities,
    'Loan_Status': loan_status
})
```

---

## 🔍 Step 2: Audit and Remediation Reference

| Issue | How to Identify | How to Fix It |
| :--- | :--- | :--- |
| **1. Skew** | **Identify:** Calculate skewness using `df['Income'].skew()`. A value **greater than 1.0** indicates a heavily right-skewed distribution. | **Fix:** Apply a **Log Transformation** (`np.log1p`) to compress extreme values closer to the mean and pull the distribution back to a normal bell curve. |
| **2. Correlation** | **Identify:** Compute a Pearson correlation matrix using `df.corr()`. Any correlation coefficient **above 0.80** implies strong multicollinearity, which confuses machine learning models. | **Fix:** Drop one of the redundant columns. Keep the feature that is more interpretable or relevant to your target variable. |
| **3. Bias** | **Identify:** Evaluate category proportions using `value_counts(normalize=True)`. High disparity indicates an unbalanced target feature. If minority class constitutes a negligible percentage, models will struggle to learn it. | **Fix:** Balance the scales by either down-sampling the majority class or artificially cloning minority records via duplication or SMOTE.|
| **4. Long-Tailed** | **Identify:** Look at the cumulative frequency distribution.  If a tiny fraction of unique categories holds the majority of rows, the feature is long-tailed. Result: New York & LA make up 63% of rows | **Fix:** Apply **Thresholding**. Keep high-volume categories as-is, and bundle all infrequent entries into an 'Other' group. |

---

## 💻 Step 3: Complete Identification & Fixing Script

Append this script right below your simulated dataset block to print out the automated data quality audit and corrections:

```python
# ==========================================
# 1. IDENTIFYING & FIXING SKEW
# ==========================================
print("--- 1. SKEW AUDIT ---")
print(f"Original Income Skewness: {df['Income'].skew():.2f}")

# Fix: Apply log-transformation
df['Income_Log'] = np.log1p(df['Income'])
print(f"Fixed Income Skewness: {df['Income_Log'].skew():.2f}\n")


# ==========================================
# 2. IDENTIFYING & FIXING CORRELATION
# ==========================================
print("--- 2. CORRELATION AUDIT ---")
correlation_coefficient = df['Experience_Years'].corr(df['Performance_Score'])
print(f"Correlation between Experience & Performance: {correlation_coefficient:.2f}")

# Fix: Drop the redundant feature to remove multi-collinearity
df_cleaned = df.drop(columns=['Performance_Score'])
print(f"Action: Dropped 'Performance_Score'. Remaining: {list(df_cleaned.columns)}\n")


# ==========================================
# 3. IDENTIFYING & FIXING TARGET BIAS
# ==========================================
print("--- 3. TARGET BIAS AUDIT ---")
print(df['Loan_Status'].value_counts(normalize=True))

# Fix: Down-sample the dominant 'Approved' class to match 'Rejected' counts
df_rejected = df[df['Loan_Status'] == 'Rejected']
df_approved_sampled = df[df['Loan_Status'] == 'Approved'].sample(n=len(df_rejected), random_state=42)

df_balanced = pd.concat([df_rejected, df_approved_sampled])
print(f"Balanced Dataset Counts:\n{df_balanced['Loan_Status'].value_counts()}\n")


# ==========================================
# 4. IDENTIFYING & FIXING LONG-TAILED VALUES
# ==========================================
print("--- 4. LONG-TAILED CATEGORICAL AUDIT ---")
print(f"Total unique cities: {df['City'].nunique()}")
print("Top 5 city distribution:\n", df['City'].value_counts().head(5))

# Fix: Group categories that appear fewer than 6 times into 'Other'
threshold = 6
city_counts = df['City'].value_counts()
rare_cities = city_counts[city_counts < threshold].index

df['City_Cleaned'] = df['City'].replace(rare_cities, 'Other')
print("\nFixed City Distribution:")
print(df['City_Cleaned'].value_counts())
```

In [9]:
# Step 4: Complete Python Code

import pandas as pd
import numpy as np

# Simple script to build data and run metrics
np.random.seed(42)
n = 100

income = np.random.lognormal(mean=10, sigma=1.0, size=n)
experience = np.random.randint(1, 25, size=n)
performance = experience * 3.5 + np.random.normal(0, 5, size=n)
loan_status = np.random.choice(['Approved', 'Rejected'], size=n, p=[0.90, 0.10])

# Long-tail categorical
cities = ['New York', 'Los Angeles', 'Chicago'] + [f'Town_{i}' for i in range(20)]
probs = [0.5, 0.2, 0.1] + [0.2/20]*20
chosen_cities = np.random.choice(cities, size=n, p=probs)

df = pd.DataFrame({
    'Income': income,
    'Experience_Years': experience,
    'Performance_Score': performance,
    'City': chosen_cities,
    'Loan_Status': loan_status
})

print(f"Skew: {df['Income'].skew():.2f}")
print(f"Corr: {df['Experience_Years'].corr(df['Performance_Score']):.2f}\n")
print(f"Skew: {df['Loan_Status'].value_counts(normalize=True)}\n")
print(f"Skew: {df['City'].value_counts().head(5)}\n")

# ==========================================
# 1. IDENTIFYING & FIXING SKEW
# ==========================================
print("--- 1. SKEW AUDIT ---")
print(f"Original Income Skewness: {df['Income'].skew():.2f}")

# Fix: Apply log-transformation
df['Income_Log'] = np.log1p(df['Income'])
print(f"Fixed Income Skewness: {df['Income_Log'].skew():.2f}\n")


# ==========================================
# 2. IDENTIFYING & FIXING CORRELATION
# ==========================================
print("--- 2. CORRELATION AUDIT ---")
correlation_coefficient = df['Experience_Years'].corr(df['Performance_Score'])
print(f"Correlation between Experience & Performance: {correlation_coefficient:.2f}")

# Fix: Drop the redundant feature to remove multi-collinearity
df_cleaned = df.drop(columns=['Performance_Score'])
print(f"Action: Dropped 'Performance_Score'. Remaining: {list(df_cleaned.columns)}\n")


# ==========================================
# 3. IDENTIFYING & FIXING TARGET BIAS
# ==========================================
print("--- 3. TARGET BIAS AUDIT ---")
print(df['Loan_Status'].value_counts(normalize=True))

# Fix: Down-sample the dominant 'Approved' class to match 'Rejected' counts
df_rejected = df[df['Loan_Status'] == 'Rejected']
df_approved_sampled = df[df['Loan_Status'] == 'Approved'].sample(n=len(df_rejected), random_state=42)

df_balanced = pd.concat([df_rejected, df_approved_sampled])
print(f"Balanced Dataset Counts:\n{df_balanced['Loan_Status'].value_counts()}\n")


# ==========================================
# 4. IDENTIFYING & FIXING LONG-TAILED VALUES
# ==========================================
print("--- 4. LONG-TAILED CATEGORICAL AUDIT ---")
print(f"Total unique cities: {df['City'].nunique()}")
print("Top 5 city distribution:\n", df['City'].value_counts().head(5))

# Fix: Group categories that appear fewer than 6 times into 'Other'
threshold = 6
city_counts = df['City'].value_counts()
rare_cities = city_counts[city_counts < threshold].index

df['City_Cleaned'] = df['City'].replace(rare_cities, 'Other')
print("\nFixed City Distribution:")
print(df['City_Cleaned'].value_counts())

Skew: 1.89
Corr: 0.99

Skew: Loan_Status
Approved    0.9
Rejected    0.1
Name: proportion, dtype: float64

Skew: City
New York       45
Los Angeles    18
Chicago         7
Town_16         5
Town_18         4
Name: count, dtype: int64

--- 1. SKEW AUDIT ---
Original Income Skewness: 1.89
Fixed Income Skewness: -0.18

--- 2. CORRELATION AUDIT ---
Correlation between Experience & Performance: 0.99
Action: Dropped 'Performance_Score'. Remaining: ['Income', 'Experience_Years', 'City', 'Loan_Status', 'Income_Log']

--- 3. TARGET BIAS AUDIT ---
Loan_Status
Approved    0.9
Rejected    0.1
Name: proportion, dtype: float64
Balanced Dataset Counts:
Loan_Status
Rejected    10
Approved    10
Name: count, dtype: int64

--- 4. LONG-TAILED CATEGORICAL AUDIT ---
Total unique cities: 19
Top 5 city distribution:
 City
New York       45
Los Angeles    18
Chicago         7
Town_16         5
Town_18         4
Name: count, dtype: int64

Fixed City Distribution:
City_Cleaned
New York       45
Other          3